#### Libraries, utils

In [240]:
import torch
import math
import numpy as np
import pickle
# -----------------------
# Activation
# -----------------------
@torch.jit.script
def phi(x):
    return torch.tanh(x)

# -----------------------
# Interaction matrix (fast, symmetric)
# -----------------------
def inter_matrix_torch(K, N, g, gamma, device):
    z0 = torch.randn(N, N, device=device)
    z1 = torch.randn(N, N, device=device)
    J = torch.zeros(N, N, device=device)
    scale = g / math.sqrt(K)

    iu = torch.triu_indices(N, N, offset=1)
    i, j = iu[0], iu[1]

    J[i, j] = scale * z0[i, j]
    J[j, i] = scale * (gamma * z0[i, j] + math.sqrt(1 - gamma**2) * z1[i, j])
    return J

# -----------------------
# Adjacency matrix (fast)
# -----------------------
def adjacency_matrix_torch(N, K, degree_sequence, device):
    p = torch.outer(degree_sequence, degree_sequence) / (K * N)
    p.clamp_(0.0, 1.0)
    A = (torch.rand(N, N, device=device) < p).float()
    A = torch.triu(A, diagonal=1)
    return A + A.T

# -----------------------
# Log-normal degree sampling
# -----------------------
def sample_lognormal_torch(mu, sigma, N, device):
    return torch.exp(mu + sigma * torch.randn(N, device=device))

# -----------------------
# Replica distance simulation
# -----------------------
def replica_distance_torch(params, hyperparams, device="cuda"):
    N, g, gamma, mu, sigma, n = params
    N_steps, dt = hyperparams

    device = torch.device(device if torch.cuda.is_available() else "cpu")

    # Degree sequence and connectivity
    degree_sequence = sample_lognormal_torch(mu, sigma, N, device)
    K = int(degree_sequence.mean().item())

    # Interaction matrices
    A = adjacency_matrix_torch(N, K, degree_sequence, device)
    J = inter_matrix_torch(K, N, g, gamma, device)
    W = A * J

    # Initial conditions
    x1 = 2 * torch.rand(N, device=device) - 1
    x2 = 2 * torch.rand(N, device=device) - 1

    # Accumulators for mean activity
    c1 = torch.zeros(N, device=device)
    c2 = torch.zeros(N, device=device)

    # Replica distance over time
    d = torch.zeros(N_steps, device=device)

    # -----------------------
    # Euler integration
    # -----------------------
    for t in range(N_steps):
        phi_x1 = phi(x1)
        phi_x2 = phi(x2)

        x1 = x1 + dt * (-x1 + torch.matmul(W, phi_x1))
        x2 = x2 + dt * (-x2 + torch.matmul(W, phi_x2))

        c1 += phi_x1
        c2 += phi_x2

        d[t] = torch.mean((c1 / (t + 1) - c2 / (t + 1)) ** 2)

    return d.cpu().numpy()

#### Replica simulations

In [33]:
import torch
import pickle

# -----------------------
# Simulation parameters
# -----------------------
N_list = [4000]
g = 3
gamma_list = [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1]
mu = 3
sigma = 1
N_samples = 100

dt = 0.1
N_steps = 100_000
hyperparams = [N_steps, dt]

# -----------------------
# Build parameter list
# -----------------------
params_list = [
    [N, g, gamma, mu, sigma, n]
    for N in N_list
    for gamma in gamma_list
    for n in range(N_samples)
]

print("Number of simulations to run:", len(params_list))

# -----------------------
# Run simulations
# -----------------------
results = []

with torch.no_grad():
    for params in params_list:
        N, g, gamma, mu, sigma, n = params

        # Compute replica distance
        d = replica_distance_torch(params, hyperparams, device="cuda")

        # Store results
        results.append({
            "N": N,
            "g": g,
            "gamma": gamma,
            "mu": mu,
            "sigma": sigma,
            "n": n,
            "d": d
        })

        print(f"Done: N={N}, gamma={gamma:.1f}, n={n}")

# -----------------------
# Save results
# -----------------------
with open("repliche_N4000.pkl", "wb") as f:
    pickle.dump(results, f)